In [1]:
import pandas as pd
import numpy as np
import re
import string

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

In [4]:
import os
print(os.getcwd())

C:\Users\delta


In [5]:
df = pd.read_csv("IMDB Dataset.csv")

In [6]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [7]:
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (50000, 2)

Columns:
['review', 'sentiment']


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [9]:
print("Unique sentiment values:")
print(df["sentiment"].unique())

Unique sentiment values:
['positive' 'negative']


In [10]:
print("Sentiment value counts:")
print(df["sentiment"].value_counts())

Sentiment value counts:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [11]:
df = df[["review", "sentiment"]]
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [12]:
duplicates_before = df.duplicated().sum()
print("Number of duplicate rows before removal:", duplicates_before)

df = df.drop_duplicates()

duplicates_after = df.duplicated().sum()
print("Number of duplicate rows after removal:", duplicates_after)
print("Shape after removing duplicates:", df.shape)

Number of duplicate rows before removal: 418
Number of duplicate rows after removal: 0
Shape after removing duplicates: (49582, 2)


In [13]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"<br\s*/?>", " ", text)  # remove HTML line breaks
    text = re.sub(r"http\S+|www\S+", " ", text)  # remove URLs
    text = re.sub(r"\d+", " ", text)  # remove numbers
    text = text.translate(str.maketrans("", "", string.punctuation))  # remove punctuation
    text = re.sub(r"\s+", " ", text).strip()  # remove extra whitespace
    return text

In [14]:
df["clean_review"] = df["review"].apply(clean_text)
df.head()

,review,sentiment,clean_review
0,One of the other reviewers has mentioned that ...,positive,one of the other reviewers has mentioned that ...
1,A wonderful little production. <br /><br />The...,positive,a wonderful little production the filming tech...
2,I thought this was a wonderful way to spend ti...,positive,i thought this was a wonderful way to spend ti...
3,Basically there's a family where a little boy ...,negative,basically theres a family where a little boy j...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,petter matteis love in the time of money is a ...


In [15]:
print("Original review:\n")
print(df["review"].iloc[0])

print("\n" + "=" * 80 + "\n")

print("Cleaned review:\n")
print(df["clean_review"].iloc[0])

Original review:

One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show

In [16]:
empty_reviews = (df["clean_review"].str.strip() == "").sum()
print("Number of empty cleaned reviews:", empty_reviews)

Number of empty cleaned reviews: 0


In [17]:
df = df[df["clean_review"].str.strip() != ""]
print("Shape after removing empty cleaned reviews:", df.shape)

Shape after removing empty cleaned reviews: (49582, 3)


In [18]:
label_encoder = LabelEncoder()
df["sentiment_encoded"] = label_encoder.fit_transform(df["sentiment"])

In [19]:
label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
print("Label mapping:", label_mapping)

Label mapping: {'negative': np.int64(0), 'positive': np.int64(1)}


In [20]:
X = df["clean_review"]
y = df["sentiment_encoded"]

In [21]:
vectorizer = TfidfVectorizer(max_features=5000)
X_vectorized = vectorizer.fit_transform(X)

In [22]:
print("Shape of TF-IDF feature matrix:", X_vectorized.shape)

Shape of TF-IDF feature matrix: (49582, 5000)


In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X_vectorized,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [24]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (39665, 5000)
X_test shape: (9917, 5000)
y_train shape: (39665,)
y_test shape: (9917,)


In [25]:
df.to_csv("cleaned_sentiment_data.csv", index=False)
print("Cleaned dataset saved as cleaned_sentiment_data.csv")

Cleaned dataset saved as cleaned_sentiment_data.csv


In [26]:
print("Preprocessing completed successfully.")
print("Final dataset shape:", df.shape)
print("Feature column used:", "clean_review")
print("Target column used:", "sentiment_encoded")
print("Train-test split: 80% train, 20% test")

Preprocessing completed successfully.
Final dataset shape: (49582, 4)
Feature column used: clean_review
Target column used: sentiment_encoded
Train-test split: 80% train, 20% test


In [27]:
df[["review", "clean_review", "sentiment", "sentiment_encoded"]].head(10)

,review,clean_review,sentiment,sentiment_encoded
0,One of the other reviewers has mentioned that ...,one of the other reviewers has mentioned that ...,positive,1
1,A wonderful little production. <br /><br />The...,a wonderful little production the filming tech...,positive,1
2,I thought this was a wonderful way to spend ti...,i thought this was a wonderful way to spend ti...,positive,1
3,Basically there's a family where a little boy ...,basically theres a family where a little boy j...,negative,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter matteis love in the time of money is a ...,positive,1
5,"Probably my all-time favorite movie, a story o...",probably my alltime favorite movie a story of ...,positive,1
6,I sure would like to see a resurrection of a u...,i sure would like to see a resurrection of a u...,positive,1
7,"This show was an amazing, fresh & innovative i...",this show was an amazing fresh innovative idea...,negative,0
8,Encouraged by the positive comments about this...,encouraged by the positive comments about this...,negative,0
9,If you like original gut wrenching laughter yo...,if you like original gut wrenching laughter yo...,positive,1


In [28]:
print("ORIGINAL REVIEW:\n")
print(df.loc[0, "review"])

print("\n" + "=" * 100 + "\n")

print("CLEANED REVIEW:\n")
print(df.loc[0, "clean_review"])

ORIGINAL REVIEW:

One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show

In [29]:
cleaned_df = df[["clean_review", "sentiment", "sentiment_encoded"]]
cleaned_df.to_csv("cleaned_sentiment_data.csv", index=False)
print("Cleaned data saved as cleaned_sentiment_data.csv")

Cleaned data saved as cleaned_sentiment_data.csv


## Preprocessing Summary

In this notebook, the sentiment analysis dataset was cleaned and prepared for machine learning. Missing values and duplicate rows were removed. The review text was cleaned by converting it to lowercase, removing HTML tags, punctuation, numbers, URLs, and extra spaces. The sentiment labels were encoded into numerical values. Finally, TF-IDF vectorization was used to transform the text into numerical features, and the dataset was split into training and testing sets using an 80:20 ratio.